In [1]:
import pandas as pd
from pathlib import Path


In [2]:
# to root tou dataset opws einai sto pc mou
DATA_ROOT = Path(r"C:\Users\georg\Desktop\MongoDB_exe - MaritimeRoutes\datasets")

#  vasikous fakelous pou tha xrhsimopoihsw sto inspection
DYN2019_DIR   = DATA_ROOT / "unipi_ais_dynamic_2019"
STATIC_DIR    = DATA_ROOT / "unipi_ais_static"
SYNOPS2019_DIR= DATA_ROOT / "unipi_ais_dynamic_synopses" / "2019"
GEODATA_DIR   = DATA_ROOT / "geodata"
WEATHER_JAN19 = DATA_ROOT / "noaa_weather" / "2019" / "jan"


In [3]:
# grhgoro list gia na dw an ontws yparxoun ta arxeia
print("Dynamic 2019:", len(list(DYN2019_DIR.glob("*.csv"))))
print("Static:", len(list(STATIC_DIR.glob("*.csv"))))
print("Synopses 2019:", len(list(SYNOPS2019_DIR.glob("*.csv"))))
print("Geodata shp:", len(list(GEODATA_DIR.glob("*.shp"))))
print("Weather Jan shp:", len(list(WEATHER_JAN19.glob("*.shp"))))


Dynamic 2019: 12
Static: 0
Synopses 2019: 0
Geodata shp: 0
Weather Jan shp: 1


In [4]:
#epilegw ena mina (jan) gia na doulepsw me mikrotero ogko
DYN_JAN = DYN2019_DIR / "unipi_ais_dynamic_jan2019.csv"
DYN_JAN


WindowsPath('C:/Users/georg/Desktop/MongoDB_exe - MaritimeRoutes/datasets/unipi_ais_dynamic_2019/unipi_ais_dynamic_jan2019.csv')

In [5]:
#  diavazw liges grammes gia na dw columns/types xwris na fortwsw olo to arxeio
dyn = pd.read_csv(DYN_JAN, nrows=5000)
dyn.head()


,t,vessel_id,lon,lat,heading,speed,course
0,1548701541000,29b2c240d3ee158281fb4ad556939ee938d08685bd796a...,23.615058,37.944708,NaN,0.0,221.3
1,1548701542000,4370e27814614fc0a8e27e9289a7a9299dc8add830e993...,23.554450,37.875880,141.0,0.3,6.4
2,1548701543000,5de00399a2fdba5f2eb761a622a79a73af9d4058d51e6b...,23.641648,37.944590,NaN,0.0,196.4
3,1548701544000,7f080c68ffb5cbe042d98b2524fc8a56db5bb75315f106...,23.641433,37.947900,179.0,0.0,103.0
4,1548701545000,0fe51bee65c87f2f06f33d5b42fd47dfaf7d99c72336e2...,23.612498,37.897965,175.0,14.5,173.5


In [6]:
#  blepw poia pedia exei to raw csv kai kanw ena grhgoro describe
print(dyn.columns.tolist())
dyn.describe(include="all")


['t', 'vessel_id', 'lon', 'lat', 'heading', 'speed', 'course']


,t,vessel_id,lon,lat,heading,speed,course
count,5.000000e+03,5000,5000.000000,5000.000000,2973.000000,4992.000000,4558.000000
unique,NaN,115,NaN,NaN,NaN,NaN,NaN
top,NaN,ae75f098caecc8c85bff774f4278e2798cf984876d656e...,NaN,NaN,NaN,NaN,NaN
freq,NaN,395,NaN,NaN,NaN,NaN,NaN
mean,1.548703e+12,NaN,23.612844,37.928985,173.617894,3.126783,169.497762
std,8.718335e+05,NaN,0.042237,0.033677,88.927648,5.913058,109.818432
min,1.548702e+12,NaN,23.391842,37.777733,0.000000,0.000000,0.000000
25%,1.548702e+12,NaN,23.588735,37.930017,153.000000,0.000000,62.925000
50%,1.548703e+12,NaN,23.615018,37.940158,178.000000,0.100000,181.000000
75%,1.548704e+12,NaN,23.641628,37.948137,214.000000,2.200000,263.300000


In [7]:
#  metatrepw to UNIX time (ms) se datetime gia na douleuei kala se queries/meta
time_col = "t" if "t" in dyn.columns else "timestamp"
dyn["dt_utc"] = pd.to_datetime(dyn[time_col], unit="ms", utc=True, errors="coerce")

dyn[[time_col, "dt_utc"]].head(10)


,t,dt_utc
0,1548701541000,2019-01-28 18:52:21+00:00
1,1548701542000,2019-01-28 18:52:22+00:00
2,1548701543000,2019-01-28 18:52:23+00:00
3,1548701544000,2019-01-28 18:52:24+00:00
4,1548701545000,2019-01-28 18:52:25+00:00
5,1548701546000,2019-01-28 18:52:26+00:00
6,1548701546000,2019-01-28 18:52:26+00:00
7,1548701547000,2019-01-28 18:52:27+00:00
8,1548701548000,2019-01-28 18:52:28+00:00
9,1548701548000,2019-01-28 18:52:28+00:00


In [8]:
#  metraw poso syxna leipei kathe pedio (NaN rate)
missing_rate = dyn.isna().mean().sort_values(ascending=False)
missing_rate.head(15)


heading      0.4054
course       0.0884
speed        0.0016
t            0.0000
lat          0.0000
lon          0.0000
vessel_id    0.0000
dt_utc       0.0000
dtype: float64

In [9]:
#  upologizw bounding box gia na dw an oi syntetagmenes einai sto swsto meros
lon_col = "lon" if "lon" in dyn.columns else "longitude"
lat_col = "lat" if "lat" in dyn.columns else "latitude"

bbox = {
    "min_lon": float(dyn[lon_col].min()),
    "max_lon": float(dyn[lon_col].max()),
    "min_lat": float(dyn[lat_col].min()),
    "max_lat": float(dyn[lat_col].max()),
}
bbox


{'min_lon': 23.3918416666667,
 'max_lon': 23.6839266666667,
 'min_lat': 37.777733333333295,
 'max_lat': 38.02941}

In [10]:
# vriskw poia ploia exoun ta perissotera points sto sample mou
vessel_col = "vessel_id" if "vessel_id" in dyn.columns else "mmsi"
dyn[vessel_col].value_counts().head(10)


vessel_id
ae75f098caecc8c85bff774f4278e2798cf984876d656ebe4a586dda38018c97    395
5ba4836f5f81948f3cd56034b22750861943cc65d311f250b8072c528736ea26    304
b9c2e66d0b02559e9cebc08025e252b7bb82837291d5b9c28991fd5bf557ad03    268
5de00399a2fdba5f2eb761a622a79a73af9d4058d51e6b16077c9c38969fb2de    255
ca4ad8573dd79dbc4b5d4be6f3cf2082178cc3a77575e84e58fe9c4d3d1caa0d    252
98bb01857390ca864cb823439f8a5f32bdb38ff27ccab30a0c930d9f2ef15273    250
29b2c240d3ee158281fb4ad556939ee938d08685bd796a6877cbecd49a5655c4    239
66d16aac6efd45973438ed0f717eb8a1d0d4d40a3fd40dec11e3b9ec40730b1e    234
26199b57ef2b787a02bf09d81b98aa7ca1a984091c6a2b6d00ee43237a2535b0    217
7f080c68ffb5cbe042d98b2524fc8a56db5bb75315f1063e6fa7440d20881f8e    209
Name: count, dtype: int64

In [11]:
# kanw sort ana ploio + xrono kai vlepo ta time gaps (gia na apofasisw threshold)
tmp = dyn.dropna(subset=["dt_utc"]).sort_values([vessel_col, "dt_utc"]).copy()

v = tmp[vessel_col].value_counts().index[0]  # top vessel sto sample
one = tmp[tmp[vessel_col] == v].copy()
one["gap_min"] = one["dt_utc"].diff().dt.total_seconds() / 60

one[["dt_utc", "gap_min"]].head(25), one["gap_min"].describe()


(                       dt_utc   gap_min
 11  2019-01-28 18:52:29+00:00       NaN
 25  2019-01-28 18:52:38+00:00  0.150000
 45  2019-01-28 18:52:49+00:00  0.183333
 63  2019-01-28 18:52:59+00:00  0.166667
 79  2019-01-28 18:53:09+00:00  0.166667
 91  2019-01-28 18:53:17+00:00  0.133333
 140 2019-01-28 18:53:47+00:00  0.500000
 157 2019-01-28 18:53:59+00:00  0.200000
 177 2019-01-28 18:54:08+00:00  0.150000
 190 2019-01-28 18:54:17+00:00  0.150000
 216 2019-01-28 18:54:29+00:00  0.200000
 232 2019-01-28 18:54:38+00:00  0.150000
 245 2019-01-28 18:54:47+00:00  0.150000
 265 2019-01-28 18:54:59+00:00  0.200000
 290 2019-01-28 18:55:17+00:00  0.300000
 312 2019-01-28 18:55:29+00:00  0.200000
 327 2019-01-28 18:55:38+00:00  0.150000
 338 2019-01-28 18:55:47+00:00  0.150000
 350 2019-01-28 18:55:59+00:00  0.200000
 365 2019-01-28 18:56:08+00:00  0.150000
 374 2019-01-28 18:56:17+00:00  0.150000
 393 2019-01-28 18:56:29+00:00  0.200000
 420 2019-01-28 18:56:47+00:00  0.300000
 459 2019-01-28 

In [13]:
#  psaxnw mesa sto STATIC_DIR gia opoiodhpote csv (giati den kswrw to akrives onoma)
static_csvs = sorted(list(STATIC_DIR.glob("*.csv")))
print("Found static csv files:", len(static_csvs))
static_csvs[:10]


Found static csv files: 0


[]

In [17]:
# orizw to swsto folder gia static
STATIC_DIR = DATA_ROOT / "ais_static"
STATIC_DIR


WindowsPath('C:/Users/georg/Desktop/MongoDB_exe - MaritimeRoutes/datasets/ais_static')

In [18]:
#  blepw ti arxeia yparxoun mesa sto ais_static
static_files = sorted(list(STATIC_DIR.iterdir()))
[(p.name, p.suffix) for p in static_files]


[('ais_codes_descriptions.csv', '.csv'),
 ('README.md', '.md'),
 ('unipi_ais_static.csv', '.csv')]

In [ ]:
# vriskw to arxeio me ta static stoixeia ploion (unipi_ais_static)
static_main = [p for p in static_files if "unipi_ais_static" in p.name.lower()]
static_main


[WindowsPath('C:/Users/georg/Desktop/MongoDB_exe - MaritimeRoutes/datasets/ais_static/unipi_ais_static.csv')]

In [20]:
# fortwnw to static arxeio (an einai xlsx -> read_excel, an einai csv -> read_csv)
if len(static_main) == 0:
    print("DEN vrhka unipi_ais_static arxeio - tsekare ta filenames sto Cell 13")
else:
    STATIC_PATH = static_main[0]
    print("Using:", STATIC_PATH.name)

    if STATIC_PATH.suffix.lower() in [".xlsx", ".xls"]:
        static_df = pd.read_excel(STATIC_PATH, nrows=20000)
    else:
        static_df = pd.read_csv(STATIC_PATH, nrows=20000)

    static_df.head()


Using: unipi_ais_static.csv


In [21]:
# koitaw pedia kai poso leipoun times (gia na dw an einai lookup)
print(static_df.columns.tolist())
static_df.isna().mean().sort_values(ascending=False).head(15)


['vessel_id', 'country', 'shiptype']


shiptype     0.015088
country      0.000482
vessel_id    0.000000
dtype: float64

In [22]:
#  anoigw to lookup arxeio me perigrafes typwn (shiptype codes)
codes_file = [p for p in static_files if "codes" in p.name.lower()]
codes_file


[WindowsPath('C:/Users/georg/Desktop/MongoDB_exe - MaritimeRoutes/datasets/ais_static/ais_codes_descriptions.csv')]

In [23]:
# fortwnw to codes file 
if len(codes_file) == 0:
    print("DEN vrhka codes file - des Cell 13")
else:
    CODES_PATH = codes_file[0]
    print("Using:", CODES_PATH.name)

    if CODES_PATH.suffix.lower() in [".xlsx", ".xls"]:
        codes_df = pd.read_excel(CODES_PATH)
    else:
        codes_df = pd.read_csv(CODES_PATH)

    codes_df.head()


Using: ais_codes_descriptions.csv


In [24]:
# kanw ena grhgoro join gia na dw an tairiazoun oi kwdikoi typwn
cand_static = [c for c in static_df.columns if "ship" in c.lower() and "type" in c.lower()]
cand_codes  = [c for c in codes_df.columns if "code" in c.lower() or ("type" in c.lower() and "ship" in c.lower())]

print("static shiptype-like cols:", cand_static)
print("codes code/type-like cols:", cand_codes)


static shiptype-like cols: ['shiptype']
codes code/type-like cols: ['Type Code']


In [25]:
# psaxnw mesa sto datasets gia folder me "synops" gia na vrw pou einai ta synopses
syn_dirs = [p for p in DATA_ROOT.rglob("*") if p.is_dir() and "synops" in p.name.lower()]
syn_dirs


[WindowsPath('C:/Users/georg/Desktop/MongoDB_exe - MaritimeRoutes/datasets/unipi_ais_dynamic_synopses'),
 WindowsPath('C:/Users/georg/Desktop/MongoDB_exe - MaritimeRoutes/datasets/unipi_ais_dynamic_synopses/ais_synopses')]

In [26]:
#  an vrw synopses folder, blepw ti arxeia exei mesa
if len(syn_dirs) == 0:
    print("DEN vrhka synopses folder - des ta folders me to Cell 21 pio katw")
else:
    SYN_DIR = syn_dirs[0]
    syn_files = sorted(list(SYN_DIR.rglob("*.csv"))) + sorted(list(SYN_DIR.rglob("*.xlsx")))
    print("Using syn folder:", SYN_DIR)
    print("Syn files:", len(syn_files))
    syn_files[:10]


Using syn folder: C:\Users\georg\Desktop\MongoDB_exe - MaritimeRoutes\datasets\unipi_ais_dynamic_synopses
Syn files: 32


In [27]:
# emfanizw ola ta subfolders mesa sto datasets gia na exw swsta paths
datasets_dir = DATA_ROOT
sorted([p.name for p in datasets_dir.iterdir() if p.is_dir()])


['ais_static',
 'geodata',
 'noaa_weather',
 'unipi_ais_dynamic_2017',
 'unipi_ais_dynamic_2018',
 'unipi_ais_dynamic_2019',
 'unipi_ais_dynamic_synopses']